# AIC-2026 — BEiT-3 keyframe embeddings

Notebook Colab độc lập cho GPU A100 40 GB. Pipeline clone project và Microsoft UNILM, xác thực Google Drive, tải đủ 873 ZIP từ hai folder, embed từng video, lưu shard có resume, rồi ghép ma trận cuối.

Notebook dùng checkpoint **BEiT-3 Base COCO Retrieval 384×384** chính thức. Đây là checkpoint ITC có projection head ảnh–text (768 chiều), phù hợp cho text-to-keyframe retrieval hơn checkpoint classification/pretraining thuần. Chọn **Runtime → Change runtime type → A100 GPU** trước khi chạy.

In [ ]:
# 1) Clone project và implementation BEiT-3 chính thức
from pathlib import Path

PROJECT_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
PROJECT_DIR = Path("/content/AIC-2026")
UNILM_DIR = Path("/content/unilm")

if not (PROJECT_DIR / ".git").exists():
    !git clone --depth 1 {PROJECT_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull --ff-only

if not (UNILM_DIR / ".git").exists():
    !git clone --depth 1 https://github.com/microsoft/unilm.git {UNILM_DIR}
else:
    !git -C {UNILM_DIR} pull --ff-only

%cd /content/AIC-2026

In [ ]:
# 2) Chỉ cài dependency inference cần thiết; bỏ legacy deepspeed/torchmetrics.
%pip install -q "timm==0.4.12" "torchscale==0.2.0" einops blobfile sentencepiece google-api-python-client google-auth-httplib2 tqdm Pillow requests

In [ ]:
# 3) Mount Drive để giữ output và cấp quyền đọc hai shared folders
from google.colab import auth, drive

drive.mount("/content/drive")
auth.authenticate_user()
print("Google Drive authentication ready")

In [ ]:
# 4) Cấu hình chạy
from pathlib import Path
import shutil
import torch

FOLDER_IDS = [
    "1ZjLlGH0Igq70wrAELVIU4kLIWSFUAN4B",  # 439 ZIP ở snapshot đã kiểm tra
    "1nxum5Qp5_iCQIqud11I8u8ACKE8OOsv5",  # 434 ZIP ở snapshot đã kiểm tra
]
DATA_DIR = Path("/content/aic_keyframes")
ZIP_CACHE_DIR = Path("/content/aic_zip_cache")
OUTPUT_DIR = Path("/content/drive/MyDrive/AIC-2026/embeddings/beit3")
CHECKPOINT_DIR = Path("/content/model_checkpoints")
BATCH_SIZE = 64                                       # 384×384; giảm còn 32 nếu GPU nhỏ hơn A100
NUM_WORKERS = 4
MAX_ZIPS = None                                       # đặt 2 để smoke test, None để chạy đủ 873 ZIP

assert torch.cuda.is_available(), "Hãy bật GPU runtime trong Colab"
free_gib = shutil.disk_usage("/content").free / 1024**3
print(torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")
print(f"Local disk còn {free_gib:.1f} GiB")
if MAX_ZIPS is None and free_gib < 40:
    raise RuntimeError("Cần tối thiểu khoảng 40 GiB local disk trống cho toàn bộ keyframe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 5) Tải dữ liệu + embed. Checkpoint retrieval được tải resume từ release chính thức.
import subprocess
import sys

SCRIPT_PATH = PROJECT_DIR / "scripts" / "colab_keyframe_embeddings.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {SCRIPT_PATH}. Hãy commit + push file scripts/colab_keyframe_embeddings.py "
        "lên đúng GitHub repo/branch rồi chạy lại cell clone."
    )

command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--model", "beit3",
    "--data-dir", str(DATA_DIR),
    "--zip-cache-dir", str(ZIP_CACHE_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--beit3-repo-dir", str(UNILM_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
for folder_id in FOLDER_IDS:
    command += ["--folder-id", folder_id]
if MAX_ZIPS is not None:
    command += ["--max-zips", str(MAX_ZIPS), "--allow-count-mismatch"]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
# 6) Kiểm tra artifact cuối
import json
import numpy as np

manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
vectors = np.load(OUTPUT_DIR / "keyframes_visual_vectors.f16.npy", mmap_mode="r")
metadata_count = sum(1 for _ in (OUTPUT_DIR / "keyframes_metadata.jsonl").open(encoding="utf-8"))
sample = vectors[np.linspace(0, len(vectors) - 1, min(1000, len(vectors)), dtype=int)].astype(np.float32)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("matrix:", vectors.shape, vectors.dtype)
print("metadata rows:", metadata_count)
print("sample norm range:", np.linalg.norm(sample, axis=1).min(), np.linalg.norm(sample, axis=1).max())
assert vectors.shape == (manifest["keyframe_count"], 768)
assert metadata_count == manifest["keyframe_count"]
assert np.isfinite(sample).all()
assert np.allclose(np.linalg.norm(sample, axis=1), 1.0, atol=2e-3)
print("✅ BEiT-3 artifacts verified")

## Output

- `shards/Lxx_Vxxx.f16.npy`: checkpoint/resume theo video.
- `keyframes_visual_vectors.f16.npy`: ma trận `(177321, 768)` khi corpus đúng snapshot hiện tại.
- `keyframes_metadata.jsonl`: ánh xạ row → video/keyframe/path.
- `drive_archives_manifest.json`: snapshot input Drive.
- `run_manifest.json`: checkpoint, preprocessing, dimension, count và GPU.

Không trộn vector BEiT-3 với MetaCLIP 2: hai model tạo hai không gian embedding khác nhau. Khi tìm kiếm text, query phải được encode bằng text tower của đúng checkpoint tương ứng.